In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. LOAD THE DATASET
# ==========================================
# Reads the local file we set up in your directory
df = pd.read_csv("Telco-Customer-Churn.csv")
print("Initial Dataset Shape:", df.shape)

# ==========================================
# 2. DATA PREPROCESSING & CLEANING
# ==========================================
# Remove customerID (not useful for prediction weights)
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)
# Convert TotalCharges to numeric and handle blank/null values
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())

# Encode all categorical text variables into numeric integers
label_encoders = {}
for column in df.columns:
    if df[column].dtype == 'object':
        le = LabelEncoder()
        df[column] = le.fit_transform(df[column])
        label_encoders[column] = le  # Saved in case inverse transforming is needed later

# ==========================================
# 3. SPLITTING AND SCALING
# ==========================================
# Split features (X) from the label target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Scale features to zero mean and unit variance for faster Neural Network convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split (80% training data, 20% validation/testing)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# ==========================================
# 4. BUILDING THE ARTIFICIAL NEURAL NETWORK (ANN)
# ==========================================
model = Sequential()

# Input Layer + First Hidden Layer (32 neurons)
model.add(Dense(32, activation='relu', input_dim=X_train.shape[1]))
model.add(Dropout(0.3))  # Dropout prevents overfitting

# Second Hidden Layer (16 neurons)
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.3))

# Output Layer (1 neuron with Sigmoid for binary probability output)
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# ==========================================
# 5. MODEL TRAINING
# ==========================================
print("\n--- Starting Model Training ---")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# ==========================================
# 6. MODEL EVALUATION
# ==========================================
print("\n--- Evaluating Model ---")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

# Generate predictions and print reports
y_pred = (model.predict(X_test) > 0.5).astype("int32")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ==========================================
# 7. EXAMPLE: PREDICT CHURN FOR A NEW CUSTOMER
# ==========================================
print("\n--- New Customer Mock Prediction Example ---")
# Feature value array matching the exact structure of your data columns
new_customer_features = np.array([[
    0,     # gender (0=Female, 1=Male)
    0,     # SeniorCitizen (0=No, 1=Yes)
    1,     # Partner (1=Yes)
    0,     # Dependents (0=No)
    1,     # tenure months
    1,     # PhoneService
    0,     # MultipleLines
    1,     # InternetService
    0,     # OnlineSecurity
    1,     # OnlineBackup
    0,     # DeviceProtection
    1,     # TechSupport
    0,     # StreamingTV
    1,     # StreamingMovies
    1,     # Contract type
    0,     # PaperlessBilling
    1,     # PaymentMethod
    50.0,  # MonthlyCharges
    200.0  # TotalCharges
]])

# The new customer features must be scaled using the SAME scaler fitted on the dataset
new_customer_scaled = scaler.transform(new_customer_features)

# Generate single evaluation probability
churn_prob = model.predict(new_customer_scaled, verbose=0)[0][0]

print(f"Churn Probability for New Customer: {churn_prob:.2f}")
print("Prediction Summary:", "⚠️ Customer is likely to Churn!" if churn_prob > 0.5 else "✅ Customer is likely to stay.")
# 1. Create the customer array
new_customer_data = [[
    0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 50.0, 200.0
]]

# 2. Convert it to a DataFrame using the exact column names your scaler remembers
feature_names = df.drop('Churn', axis=1).columns
new_customer_df = pd.DataFrame(new_customer_data, columns=feature_names)

# 3. Scale and predict (No more warning!)
new_customer_scaled = scaler.transform(new_customer_df)
churn_prob = model.predict(new_customer_scaled)[0][0]

print(f"\nChurn Probability for New Customer: {churn_prob:.2f}")
print("Prediction:", "Churn" if churn_prob > 0.5 else "No Churn")

Initial Dataset Shape: (200, 21)

--- Starting Model Training ---
Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 69ms/step - accuracy: 0.6094 - loss: 0.6728 - val_accuracy: 0.6562 - val_loss: 0.6254
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6562 - loss: 0.6391 - val_accuracy: 0.6562 - val_loss: 0.6246
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6797 - loss: 0.6360 - val_accuracy: 0.6562 - val_loss: 0.6250
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7422 - loss: 0.6123 - val_accuracy: 0.6875 - val_loss: 0.6258
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7109 - loss: 0.6291 - val_accuracy: 0.6875 - val_loss: 0.6268
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7031 - loss: 0.6267 - val_accuracy: 0.6562 - val_loss: 0.6273
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.6875 - loss: 0.6109 - val_accuracy: 0.6562 - val_loss: 0.6285
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy

Test Accuracy: 70.00%
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step

Classification Report:
               precision    recall  f1-score   support

           0       0.70      1.00      0.82        28
           1       0.00      0.00      0.00        12

    accuracy                           0.70        40
   macro avg       0.35      0.50      0.41        40
weighted avg       0.49      0.70      0.58        40

Confusion Matrix:
 [[28  0]
 [12  0]]

--- New Customer Mock Prediction Example ---
Churn Probability for New Customer: 0.21
Prediction Summary: ✅ Customer is likely to stay.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step

Churn Probability for New Customer: 0.21
Prediction: No Churn


In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. FETCH AND LOAD THE DATASET
# ==========================================
# Fetch the dataset directly from scikit-learn
california = fetch_california_housing(as_frame=True)

# Create the features DataFrame and add the target variable (MedHouseVal)
df = california.frame
print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

# ==========================================
# 2. SPLITTING AND SCALING
# ==========================================
# Features (X) and Target (y)
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']  # Median house value in $100,000s

# Train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (essential for stable neural network training)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. BUILDING THE REGRESSION ANN
# ==========================================
model = Sequential()

# Input Layer + First Hidden Layer
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dropout(0.2))

# Second Hidden Layer
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

# Output Layer (1 neuron, NO activation function because this is Regression)
model.add(Dense(1, activation='linear'))

# Compile model using Mean Squared Error (MSE) for loss
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])

# ==========================================
# 4. MODEL TRAINING
# ==========================================
print("\n--- Starting Model Training ---")
history = model.fit(
    X_train_scaled, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# ==========================================
# 5. MODEL EVALUATION
# ==========================================
print("\n--- Evaluating Model Performance ---")
# Predict on the test set
y_pred = model.predict(X_test_scaled).flatten()

# Calculate regression evaluation metrics
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f} (Avg error in $100k blocks)")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared Score (R2): {r2:.4f} (Closer to 1.0 is better)")

# ==========================================
# 6. MAKE A PREDICTION ON A NEW HOUSE BLOCK
# ==========================================
print("\n--- Example: Predict Value for a New House Block ---")
# Example features: MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude
new_house_features = np.array([[
    4.5,   # MedInc: Median income in block
    25.0,  # HouseAge: Median house age
    5.5,   # AveRooms: Average number of rooms
    1.0,   # AveBedrms: Average number of bedrooms
    1200,  # Population: Block population
    2.8,   # AveOccup: Average house occupancy
    34.05, # Latitude
    -118.24# Longitude
]])

# Preprocess the sample using the fitted scaler
new_house_scaled = scaler.transform(new_house_features)

# Generate prediction
predicted_val = model.predict(new_house_scaled, verbose=0)[0][0]
print(f"Predicted Median House Value: ${predicted_val * 100000:,.2f}")

Dataset Shape: (20640, 9)

First 5 Rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

--- Starting Model Training ---
Epoch 1/40
413/413 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.4607 - mean_absolute_error: 0.7603 - val_loss: 0.5426 - val_mean_absolute_error: 0.5183
Epoch 2/40
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6630 - mean_absolute_error: 0.5813 - val_loss: 0.4563 - val_mean_absolute_error: 0.4710
E